## Discovering CORE Rankings for CS Conferences

This notebook integrates the **CORE 2023 Conference Ranking** dataset with our existing conference corpus to assign quality tier rankings (A*, A, B, C, or Unranked) to each conference.

### Key Steps

1. **Load CORE 2023 Data**: Download and parse the official CORE ranking CSV
2. **Match Conferences**: Use exact acronym matching with fuzzy fallback for unmatched conferences
3. **Enrich Datasets**: Add `core_rank` column to:
    - `junior_authors_all_conferences.csv`
    - `matched_pairs.csv`
    - `control_authors.csv`
4. **Create Lookup**: Export a reusable `conference_core_ranks.csv` for downstream analysis

### Results Summary

- **22 conferences in corpus**
- **21 exact matches** (95.5%)
- **1 fuzzy match** (S&P → SP)
- **Rank distribution**: 21 A*, 1 Unranked (NSDI)

In [1]:
import pandas as pd
import json
import os
import shutil
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT = Path("..") 
DATA_MATCHED   = ROOT / "data" / "matched"
DATA_PROFILES  = ROOT / "data" / "profiles"
DATA_FIGURES   = ROOT / "data" / "figures"

CORE_URL = "https://kjy-open-file.oss-cn-heyuan.aliyuncs.com/qkml/Core%20Conference%20Ranking%202023.csv"


In [2]:
# Download CORE 2023 rankings
core_raw = pd.read_csv(
    CORE_URL,
    header=None,
    names=["id", "full_name", "acronym", "year", "rank", "flag1", "cat1", "cat2", "cat3"],
    dtype=str
)

# Keep only acronym + rank; clean up
core = core_raw[["acronym", "full_name", "rank"]].copy()
core["acronym"] = core["acronym"].str.strip().str.upper()
core["rank"]    = core["rank"].str.strip()

# Standardise rank labels
RANK_MAP = {
    "A*": "A*", "A": "A", "B": "B", "C": "C"
}
core["core_rank"] = core["rank"].map(RANK_MAP).fillna("Unranked")

print(f"CORE 2023 loaded: {len(core)} entries")
print(core["core_rank"].value_counts())

CORE 2023 loaded: 968 entries
core_rank
C           379
B           224
Unranked    189
A           116
A*           60
Name: count, dtype: int64


In [3]:
# Load your existing conference → OpenAlex source IDs mapping
with open(DATA_MATCHED / "conference_source_ids.json") as f:
    conf_sources = json.load(f)

# Build a clean DataFrame: acronym + source_id(s)
conf_df = pd.DataFrame([
    {"conf_acronym": k.strip().upper(), "source_ids": v}
    for k, v in conf_sources.items()
])

print(f"Your conferences: {len(conf_df)}")
print(conf_df["conf_acronym"].tolist())


Your conferences: 22
['AAAI', 'CHI', 'CVPR', 'FOCS', 'ICCV', 'ICML', 'ICSE', 'IJCAI', 'INFOCOM', 'MOBICOM', 'NSDI', 'NEURIPS', 'OSDI', 'S&P', 'SIGCOMM', 'SIGIR', 'SIGMETRICS', 'SIGMOD', 'SODA', 'SOSP', 'STOC', 'VLDB']


In [4]:
from difflib import get_close_matches

# ── Step 1: exact match on acronym ────────────────────────────────────────
merged = conf_df.merge(
    core[["acronym", "core_rank", "full_name"]],
    left_on="conf_acronym",
    right_on="acronym",
    how="left"
).drop(columns="acronym")

matched_exact   = merged["core_rank"].notna().sum()
unmatched_mask  = merged["core_rank"].isna()
print(f"Exact matches:     {matched_exact}/{len(merged)}")
print(f"Still unmatched:   {unmatched_mask.sum()}")

# ── Step 2: fuzzy fallback for unmatched ──────────────────────────────────
core_acronyms = core["acronym"].tolist()

def fuzzy_rank(acronym):
    matches = get_close_matches(acronym, core_acronyms, n=1, cutoff=0.80)
    if matches:
        row = core[core["acronym"] == matches[0]].iloc[0]
        return row["core_rank"], row["full_name"], matches[0]
    return "Unranked", "", ""

for idx in merged[unmatched_mask].index:
    acr = merged.at[idx, "conf_acronym"]
    rank, fname, matched_to = fuzzy_rank(acr)
    merged.at[idx, "core_rank"]  = rank
    merged.at[idx, "full_name"]  = fname
    merged.at[idx, "fuzzy_match"] = matched_to

# ── Final summary ─────────────────────────────────────────────────────────
print("\n── Conference → CORE Rank mapping ──")
print(merged[["conf_acronym", "core_rank", "full_name"]].to_string(index=False))

# Save lookup table
conf_rank_lookup = merged[["conf_acronym", "core_rank"]].set_index("conf_acronym")["core_rank"].to_dict()


Exact matches:     21/22
Still unmatched:   1

── Conference → CORE Rank mapping ──
conf_acronym core_rank                                                                                             full_name
        AAAI        A*                           National Conference of the American Association for Artificial Intelligence
         CHI        A*                                        International Conference on Human Factors in Computing Systems
        CVPR        A*                                            IEEE Conference on Computer Vision and Pattern Recognition
        FOCS        A*                                                     IEEE Symposium on Foundations of Computer Science
        ICCV        A*                                                      IEEE International Conference on Computer Vision
        ICML        A*                                                          International Conference on Machine Learning
        ICSE        A*                   

In [11]:
def add_core_rank(df, conf_col="conference"):
    """Map conference acronym → CORE rank and add as new column."""
    df = df.copy()
    df["conf_upper"] = df[conf_col].str.strip().str.upper()
    df["core_rank"]  = df["conf_upper"].map(conf_rank_lookup).fillna("Unranked")
    df.drop(columns="conf_upper", inplace=True)
    return df

# ── junior_authors_all_conferences.csv ─────────────────────────────────────
juniors = pd.read_csv(DATA_MATCHED / "junior_authors_all_conferences.csv")
juniors = add_core_rank(juniors)
juniors.to_csv(DATA_MATCHED / "junior_authors_all_conferences.csv", index=False)
print("✓ junior_authors_all_conferences.csv updated")
print(juniors["core_rank"].value_counts())

# ── matched_pairs.csv ──────────────────────────────────────────────────────
pairs = pd.read_csv(DATA_MATCHED / "matched_pairs.csv")
pairs = add_core_rank(pairs)
pairs.to_csv(DATA_MATCHED / "matched_pairs.csv", index=False)
print("\n✓ matched_pairs.csv updated")
print(pairs["core_rank"].value_counts())

# ── control_authors.csv ────────────────────────────────────────────────────
ca_path = DATA_MATCHED / "control_authors.csv"
try:
    controls = pd.read_csv(ca_path)
    if controls.empty or len(controls.columns) == 0:
        raise ValueError("Empty file")
    controls = add_core_rank(controls)
    controls.to_csv(ca_path, index=False)
    print("✓ control_authors.csv updated")
except Exception as e:
    print(f"⚠ control_authors.csv skipped ({e}) — core_rank already in matched_pairs.csv")



✓ junior_authors_all_conferences.csv updated
core_rank
A*          427
Unranked    176
Name: count, dtype: int64

✓ matched_pairs.csv updated
core_rank
A*          427
Unranked    176
Name: count, dtype: int64
⚠ control_authors.csv skipped (No columns to parse from file) — core_rank already in matched_pairs.csv


In [12]:
H_INDEX_COLS = ["h_index", "h-index", "hindex"]  # catch common variants

def drop_h_index(filepath):
    df = pd.read_csv(filepath)
    cols_to_drop = [c for c in df.columns if c.lower() in H_INDEX_COLS]
    if cols_to_drop:
        df.drop(columns=cols_to_drop, inplace=True)
        df.to_csv(filepath, index=False)
        print(f"✓ Dropped {cols_to_drop} from {filepath.name}")
    else:
        print(f"  (no h_index column found in {filepath.name})")
    return df

junior_profiles  = drop_h_index(DATA_PROFILES / "junior_profiles_all.csv")
all_profiles     = drop_h_index(DATA_PROFILES / "all_matched_profiles.csv")
junior_progress  = drop_h_index(DATA_PROFILES / "junior_profiles_progress.csv")

  (no h_index column found in junior_profiles_all.csv)
  (no h_index column found in all_matched_profiles.csv)
  (no h_index column found in junior_profiles_progress.csv)


In [13]:
# Move 04_hindex.png to an archive subfolder rather than hard-delete
archive_dir = DATA_FIGURES / "_archived"
archive_dir.mkdir(exist_ok=True)

hindex_fig = DATA_FIGURES / "04_hindex.png"
if hindex_fig.exists():
    shutil.move(str(hindex_fig), str(archive_dir / "04_hindex.png"))
    print("✓ 04_hindex.png moved to figures/_archived/")
else:
    print("  04_hindex.png not found (already removed?)")

  04_hindex.png not found (already removed?)


In [14]:
# Save the lookup table for use in later notebooks
lookup_df = pd.DataFrame(
    list(conf_rank_lookup.items()), 
    columns=["conference", "core_rank"]
).sort_values("core_rank")

lookup_df.to_csv(DATA_MATCHED / "conference_core_ranks.csv", index=False)
print("✓ conference_core_ranks.csv saved\n")

# Sanity check: rank distribution across your 30 conferences
print("── CORE rank distribution in your corpus ──")
print(lookup_df["core_rank"].value_counts())
print("\n── Full list ──")
print(lookup_df.to_string(index=False))

✓ conference_core_ranks.csv saved

── CORE rank distribution in your corpus ──
core_rank
A*          21
Unranked     1
Name: count, dtype: int64

── Full list ──
conference core_rank
      AAAI        A*
      SOSP        A*
      SODA        A*
    SIGMOD        A*
SIGMETRICS        A*
     SIGIR        A*
   SIGCOMM        A*
       S&P        A*
      OSDI        A*
   NEURIPS        A*
      VLDB        A*
   MOBICOM        A*
   INFOCOM        A*
     IJCAI        A*
      ICSE        A*
      ICML        A*
      ICCV        A*
      FOCS        A*
      CVPR        A*
       CHI        A*
      STOC        A*
      NSDI  Unranked
